# Create dataset — Option B ghost cells (inflow + outflow)

Full pipeline for the **Option B** warmstart dataset:

1. Build `template_100m_inflow_outflow_gc.pkl` with two types of ghost cells:
   - **Inflow ghost cells** (msk==2): dual edges `[ghost → interior]`, prescribed WD from SFINCS
   - **Outflow ghost cells** (msk==3): dual edges `[interior → ghost]`, free-drainage (no BC)
2. Convert the warmstart SFINCS simulation to the warmstart dataset

Expected output:
```
database/datasets/train/template_100m_inflow_outflow_gc.pkl
database/datasets/train/ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_inflow_outflow_gc.pkl
database/datasets/test/ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_inflow_outflow_gc.pkl
```

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

## Config

In [ ]:
# --- Paths ---
SFINCS_DIR_TEMPLATE = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon'
)
SFINCS_MAP_WARMSTART = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart/sfincs_map.nc'
)
SFINCS_SRC_WARMSTART = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart/sfincs.src'
)

TEMPLATE_PKL = 'database/datasets/train/template_100m_inflow_outflow_gc.pkl'
DATASET_NAME = 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_inflow_outflow_gc'
OUT_ROOT     = 'database/datasets'

MESH_RESOLUTIONS = [2000, 1000, 500]   # coarse gmsh levels [m], coarsest → finest

# The existing template (if any) was built with inward edges for msk==3 cells.
# Must rebuild so ghost cells get the correct OUTWARD dual edges.
FORCE_REBUILD_TEMPLATE = True
FORCE_REBUILD_DATASET  = True

## Step 1 — Build template

In [ ]:
from database.create_mesh_template_marg import create_mesh_template_pkl

if FORCE_REBUILD_TEMPLATE and os.path.exists(TEMPLATE_PKL):
    os.remove(TEMPLATE_PKL)
    print('Deleted stale template:', TEMPLATE_PKL)

if os.path.exists(TEMPLATE_PKL):
    print('Template already exists:', TEMPLATE_PKL)
    print('Set FORCE_REBUILD_TEMPLATE = True to recreate.')
else:
    create_mesh_template_pkl(
        shapefile_path        = f'{SFINCS_DIR_TEMPLATE}/gis/region.geojson',
        dem_tif_path          = f'{SFINCS_DIR_TEMPLATE}/gis/dep.tif',
        output_pkl_path       = TEMPLATE_PKL,
        with_multiscale       = True,
        number_of_multiscales = 4,
        mesh_resolutions      = MESH_RESOLUTIONS,
        sfincs_map_nc         = f'{SFINCS_DIR_TEMPLATE}/sfincs_map.nc',
    )
    print('\nTemplate saved:', TEMPLATE_PKL)

## Step 2 — Inspect template (ghost cell counts)

In [ ]:
import pickle, torch
import numpy as np

with open(TEMPLATE_PKL, 'rb') as f:
    template_data = pickle.load(f)[0]

# After mesh_list[::-1] in template creation, SFINCS (finest) is at meshes[0].
finest = template_data.mesh.meshes[0]
finest_offset = int(template_data.finest_offset) if hasattr(template_data, 'finest_offset') \
                else int(template_data.node_ptr[-2])

print(f'Total mesh faces (finest/SFINCS): {finest.face_x.shape[0]}')
print(f'Finest offset: {finest_offset}')

# Diagnostic: raw ghost cell counts from the mesh object
n_gc_in  = len(finest.ghost_cells_ids_inflow)  if hasattr(finest, 'ghost_cells_ids_inflow')  else 'N/A'
n_gc_out = len(finest.ghost_cells_ids_outflow) if hasattr(finest, 'ghost_cells_ids_outflow') else 'N/A'
print(f'\nGhost cells (mesh): inflow={n_gc_in}  outflow={n_gc_out}')
print(f'  → inflow ghost cells  [ghost → interior]  (inward edges, prescribed WD BC)')
print(f'  → outflow ghost cells [interior → ghost]  (outward edges — currently reverted to inward)')

n_in  = len(template_data.node_BC)
n_out = len(template_data.node_BC_outflow) if hasattr(template_data, 'node_BC_outflow') else 0
print(f'\nTemplate node_BC (inflow BC):   {n_in}  — prescribed BC in training')
print(f'  → IDs: {template_data.node_BC.tolist()[:10]}{"..." if n_in > 10 else ""}')
print(f'Template node_BC_outflow:        {n_out}  — NOT in BC, evolve freely')
if n_out > 0:
    print(f'  → IDs (first 10): {template_data.node_BC_outflow.tolist()[:10]}')
print(f'BC dummy shape: {tuple(template_data.BC.shape)}')

# Ghost cell positions
if n_in > 0:
    in_local = (template_data.node_BC.numpy() - finest_offset)
    in_xy    = finest.face_xy[in_local]
    print(f'\nInflow ghost positions  (x range): {in_xy[:,0].min():.0f} – {in_xy[:,0].max():.0f}')
else:
    print('\nNo inflow ghost cells — inflow BC will use face_bnd cells near sfincs.src.')

if n_out > 0:
    out_local = (template_data.node_BC_outflow.numpy() - finest_offset)
    out_xy    = finest.face_xy[out_local]
    print(f'Outflow ghost positions (x range): {out_xy[:,0].min():.0f} – {out_xy[:,0].max():.0f}')

# Sanity
if hasattr(template_data, 'node_BC_outflow'):
    overlap = set(template_data.node_BC.tolist()) & set(template_data.node_BC_outflow.tolist())
    print(f'\nOverlap inflow ∩ outflow ghost cells (should be 0): {len(overlap)}')

### Step 2b — Visualise mesh scales + ghost cells

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

meshes = template_data.mesh.meshes   # [0] = SFINCS (finest), [1..] = gmsh coarser

# Collect positions per scale (exclude ghost cells which are appended at the end)
scale_xy    = []
scale_names = []
for i, m in enumerate(meshes):
    xy   = np.asarray(m.face_xy)
    n_gc = len(m.ghost_cells_ids) if hasattr(m, 'ghost_cells_ids') else 0
    n_reg = xy.shape[0] - n_gc
    scale_xy.append(xy[:n_reg])
    res = ['SFINCS 100m', 'gmsh ~500m', 'gmsh ~1000m', 'gmsh ~2000m']
    scale_names.append(f'{res[i] if i < len(res) else f"scale {i}"} ({n_reg:,} faces)')

# Ghost cell positions (ghost cells are appended → face_xy[local_gc_id] works directly)
finest_xy = np.asarray(meshes[0].face_xy)

gc_in_xy, gc_out_xy = None, None
n_in  = len(template_data.node_BC)
n_out = len(template_data.node_BC_outflow) if hasattr(template_data, 'node_BC_outflow') else 0

if n_in > 0:
    in_loc   = template_data.node_BC.numpy() - finest_offset
    gc_in_xy = finest_xy[in_loc]

if n_out > 0:
    out_loc   = template_data.node_BC_outflow.numpy() - finest_offset
    gc_out_xy = finest_xy[out_loc]

# ---- Figure layout ----
has_zoom = gc_in_xy is not None or gc_out_xy is not None
n_zoom   = (gc_in_xy is not None) + (gc_out_xy is not None)
fig = plt.figure(figsize=(18, 8) if n_zoom else (10, 8))

if n_zoom:
    gs = gridspec.GridSpec(n_zoom, 2, figure=fig, width_ratios=[1.6, 1], hspace=0.35, wspace=0.08)
    ax_main = fig.add_subplot(gs[:, 0])
    zoom_axes = [fig.add_subplot(gs[i, 1]) for i in range(n_zoom)]
else:
    ax_main = fig.add_subplot(111)
    zoom_axes = []

# ---- Overview ----
colors = ['#d4d4d4', '#9dbfde', '#5d93c7', '#2a6099']
sizes  = [0.5,       4,         6,          8]
alphas = [0.4,       0.7,       0.8,        0.9]

for i, (xy, name) in enumerate(zip(scale_xy, scale_names)):
    ax_main.scatter(xy[:, 0], xy[:, 1],
                    s=sizes[min(i, len(sizes)-1)], c=colors[min(i, len(colors)-1)],
                    alpha=alphas[min(i, len(alphas)-1)], label=name, rasterized=True)

if gc_in_xy is not None:
    ax_main.scatter(gc_in_xy[:, 0], gc_in_xy[:, 1],
                    s=60, c='red', marker='^', zorder=7,
                    label=f'Inflow GC — msk==2 ({n_in} cells)')
if gc_out_xy is not None:
    ax_main.scatter(gc_out_xy[:, 0], gc_out_xy[:, 1],
                    s=20, c='blue', marker='v', zorder=7,
                    label=f'Outflow GC — msk==3 ({n_out} cells)')

ax_main.set_aspect('equal')
ax_main.set_title('Multiscale mesh — all scales + ghost cells', fontsize=11)
ax_main.set_xlabel('x [m]'); ax_main.set_ylabel('y [m]')
ax_main.legend(fontsize=7, markerscale=2, loc='upper right')

# ---- Zoom panels ----
zoom_idx = 0
pad = 5000  # m padding around ghost cells

def _zoom(ax, gc_xy, bg_xy, color, marker, label):
    ax.scatter(bg_xy[:, 0], bg_xy[:, 1], s=1, c='#d4d4d4', alpha=0.5, rasterized=True)
    ax.scatter(gc_xy[:, 0], gc_xy[:, 1], s=60, c=color, marker=marker, zorder=6, label=label)
    xlo, xhi = gc_xy[:, 0].min() - pad, gc_xy[:, 0].max() + pad
    ylo, yhi = gc_xy[:, 1].min() - pad, gc_xy[:, 1].max() + pad
    ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)
    ax.set_aspect('equal'); ax.legend(fontsize=8, markerscale=1.5)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

if gc_in_xy is not None and zoom_axes:
    ax_z = zoom_axes[zoom_idx]; zoom_idx += 1
    _zoom(ax_z, gc_in_xy, scale_xy[0], 'red', '^', f'Inflow GC ({n_in} cells)')
    ax_z.set_title('Inflow ghost cells (msk==2)', fontsize=10)

if gc_out_xy is not None and zoom_idx < len(zoom_axes):
    ax_z = zoom_axes[zoom_idx]
    _zoom(ax_z, gc_out_xy, scale_xy[0], 'blue', 'v', f'Outflow GC ({n_out} cells)')
    ax_z.set_title('Outflow ghost cells (msk==3)', fontsize=10)

plt.suptitle('Template ghost cells', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print(f'Inflow GC positions:\n{gc_in_xy}' if gc_in_xy is not None else 'No inflow GC.')
if gc_out_xy is not None:
    print(f'Outflow GC x-range: {gc_out_xy[:,0].min():.0f} – {gc_out_xy[:,0].max():.0f} m')
    print(f'Outflow GC y-range: {gc_out_xy[:,1].min():.0f} – {gc_out_xy[:,1].max():.0f} m')

## Step 3 — Build warmstart dataset

In [ ]:
import copy
import xarray as xr
from scipy.interpolate import griddata
from scipy.spatial import cKDTree

from database.convert_sfincs_to_pkl_marg import (
    load_single_data_object, get_target_points, build_output_data, parse_src_file,
)

out_paths = {split: os.path.join(OUT_ROOT, split, f'{DATASET_NAME}.pkl') for split in ('train', 'test')}

if not FORCE_REBUILD_DATASET and all(os.path.exists(p) for p in out_paths.values()):
    print('Dataset already exists. Set FORCE_REBUILD_DATASET = True to recreate.')
else:
    print('Loading template...')
    template_data = load_single_data_object(TEMPLATE_PKL)
    target_points = get_target_points(template_data)
    num_targets   = target_points.shape[0]
    # After mesh_list[::-1] in template creation, SFINCS (finest) is at meshes[0].
    finest_mesh   = template_data.mesh.meshes[0]
    finest_offset = int(template_data.finest_offset) if hasattr(template_data, 'finest_offset') \
                    else int(template_data.node_ptr[-2])
    print(f'  Target mesh faces: {num_targets}  |  finest_offset: {finest_offset}')

    inflow_gc_global = template_data.node_BC.numpy().astype(np.int64)
    n_inflow_gc = len(inflow_gc_global)
    print(f'  Inflow ghost cells in template (msk==2): {n_inflow_gc}')

    print('Opening SFINCS warmstart map...')
    ds  = xr.open_dataset(SFINCS_MAP_WARMSTART, decode_times=False)
    msk = ds['msk'].values
    zs  = ds['zs'].values
    zb  = ds['zb'].values

    if msk.ndim == 1:
        msk = msk.reshape(zs.shape[1], zs.shape[2])

    x = ds.coords['x'].values
    y = ds.coords['y'].values
    if x.ndim == 1 and y.ndim == 1:
        if len(x) == msk.shape[1] and len(y) == msk.shape[0]:
            x, y = np.meshgrid(x, y, indexing='xy')
        elif x.shape[0] == msk.size:
            x = x.reshape(msk.shape)
            y = y.reshape(msk.shape)

    active        = msk > 0
    source_points = np.column_stack([x[active], y[active]])
    active_flat   = np.flatnonzero(active.reshape(-1))
    time_var      = ds.coords.get('time', ds.coords.get('t', None))
    map_times_s   = (time_var.values.astype(np.float64) if time_var is not None
                     else np.arange(zs.shape[0]) * 3600.0)
    time_steps    = zs.shape[0]
    print(f'  Active SFINCS cells: {source_points.shape[0]}  |  time steps: {time_steps}')

    # WD for full mesh
    print('Interpolating WD...')
    zs_active = zs[:, active]
    zb_active = zb[active]
    WD_active  = np.maximum(np.where(np.isnan(zs_active), zb_active, zs_active) - zb_active, 0.0).astype(np.float32)
    WD = np.zeros((num_targets, time_steps), dtype=np.float32)
    for t in range(time_steps):
        iv = griddata(source_points, WD_active[t], target_points, method='linear')
        WD[:, t] = np.nan_to_num(iv, nan=0.0)
    del WD_active, zs_active
    print(f'  WD done: {WD.shape}  peak={WD.max():.3f} m')

    # Velocities
    ds_raw = xr.open_dataset(SFINCS_MAP_WARMSTART, decode_times=False, mask_and_scale=False)
    VX = np.zeros((num_targets, time_steps), dtype=np.float32)
    VY = np.zeros((num_targets, time_steps), dtype=np.float32)
    for var, arr_out in [('u', VX), ('v', VY)]:
        if var in ds_raw.data_vars:
            print(f'Interpolating {var}...')
            raw = ds_raw[var].values.astype(np.float32)
            fv  = ds_raw[var].attrs.get('_FillValue', None)
            if fv is not None:
                raw[raw == float(fv)] = np.nan
            act = raw[:, active]
            del raw
            for t in range(time_steps):
                ok = np.isfinite(act[t])
                if ok.sum() > 3:
                    iv = griddata(source_points[ok], act[t][ok], target_points, method='linear')
                    arr_out[:, t] = np.nan_to_num(iv, nan=0.0)
            del act
    ds_raw.close()

    # Build BC
    zs_flat = zs.reshape(time_steps, -1)
    zb_flat = zb.reshape(-1)
    all_xy  = np.column_stack([x.reshape(-1), y.reshape(-1)])

    if n_inflow_gc > 0:
        print('\nBuilding BC for inflow ghost cells (msk==2 mirrors)...')
        inflow_gc_local = (inflow_gc_global - finest_offset).astype(np.int64)
        inflow_gc_xy    = np.asarray(finest_mesh.face_xy)[inflow_gc_local]

        msk2_flat = msk.reshape(-1) == 2
        n_msk2    = msk2_flat.sum()
        print(f'  SFINCS msk==2 cells in warmstart map: {n_msk2}')

        if n_msk2 > 0:
            msk2_xy   = np.column_stack([all_xy[msk2_flat, 0], all_xy[msk2_flat, 1]])
            _, gc2ref = cKDTree(msk2_xy).query(inflow_gc_xy)
            ref_wd    = np.maximum(
                np.nan_to_num(zs_flat[:, msk2_flat], nan=0.0) - zb_flat[msk2_flat], 0.0
            ).astype(np.float32)
            wd_inflow = ref_wd[:, gc2ref]
            print(f'  Source: msk==2 cells → peak WD {wd_inflow.max():.3f} m')
        else:
            print('  Warmstart map has no msk==2 — mapping ghost cells to nearest active cell.')
            _, gc2sp  = cKDTree(source_points).query(inflow_gc_xy)
            gc2flat   = active_flat[gc2sp]
            wd_inflow = np.maximum(
                np.nan_to_num(zs_flat[:, gc2flat], nan=0.0) - zb_flat[gc2flat], 0.0
            ).astype(np.float32)
            print(f'  Source: nearest active cell → peak WD {wd_inflow.max():.3f} m')

        node_bc_out = inflow_gc_global.astype(np.int32)
        n_inflow    = n_inflow_gc

    else:
        print('\nBuilding BC from face_bnd cells near sfincs.src (no msk==2 ghost cells)...')
        src_xy          = parse_src_file(SFINCS_SRC_WARMSTART)
        face_bnd_local  = np.asarray(finest_mesh.face_bnd)
        face_bnd_global = (face_bnd_local + finest_offset).astype(np.int64)
        bnd_face_xy     = np.asarray(finest_mesh.face_xy)[face_bnd_local]
        _, local_idx    = cKDTree(bnd_face_xy).query(src_xy)
        inflow_global   = face_bnd_global[local_idx]
        inflow_face_xy  = bnd_face_xy[local_idx]
        n_inflow        = len(inflow_global)
        print(f'  sfincs.src locations: {src_xy.shape[0]}  →  face_bnd inflow cells: {n_inflow}')
        _, sfincs_idx = cKDTree(all_xy).query(inflow_face_xy)
        wd_inflow     = np.maximum(
            np.nan_to_num(zs_flat[:, sfincs_idx], nan=0.0) - zb_flat[sfincs_idx], 0.0
        ).astype(np.float32)
        print(f'  Inflow WD peak: {wd_inflow.max(0).round(2)} m')
        node_bc_out = inflow_global.astype(np.int32)

    del zs, zb, zs_flat, zb_flat
    ds.close()

    bc_all = np.zeros((n_inflow, time_steps, 2), dtype=np.float32)
    bc_all[:, :, 1] = wd_inflow.T

    data_out = build_output_data(template_data, WD=WD, VX=VX, VY=VY, map_times_s=map_times_s)
    data_out.node_BC        = torch.tensor(node_bc_out, dtype=torch.int32)
    data_out.BC             = torch.FloatTensor(bc_all)
    data_out.type_BC        = torch.tensor(1, dtype=torch.int32)
    data_out.edge_BC_length = torch.ones(1, dtype=torch.float32)
    if hasattr(template_data, 'node_BC_outflow'):
        data_out.node_BC_outflow = template_data.node_BC_outflow

    print(f'\n=== BC summary ===')
    print(f'  node_BC shape : {tuple(data_out.node_BC.shape)}')
    print(f'  BC shape      : {tuple(data_out.BC.shape)}')
    print(f'  type_BC       : {data_out.type_BC.item()} (1 = fixed WD)')
    print(f'  inflow WD peak: {bc_all[:,:,1].max():.3f} m')
    if hasattr(data_out, 'node_BC_outflow'):
        print(f'  outflow ghost cells (free, not in BC): {len(data_out.node_BC_outflow)}')

    for split in ('train', 'test'):
        out_dir = os.path.join(OUT_ROOT, split)
        os.makedirs(out_dir, exist_ok=True)
        out_path = out_paths[split]
        with open(out_path, 'wb') as f:
            pickle.dump([copy.deepcopy(data_out)], f)
        print(f'  Saved: {out_path}')

    print(f'\nDone.  WD={tuple(data_out.WD.shape)}')

## Step 4 — Quick sanity check

In [ ]:
with open(out_paths['train'], 'rb') as f:
    d = pickle.load(f)[0]

print('WD shape :', tuple(d.WD.shape))
print('node_BC  :', d.node_BC.tolist())
print('BC shape :', tuple(d.BC.shape))
print('type_BC  :', d.type_BC.item(), ' (1=fixed WD)')
print('BC peak WD (channel 1):', d.BC[:, :, 1].max().item(), 'm')
print()
print('WD peak in full mesh  :', d.WD.max().item(), 'm')
print('WD min in full mesh   :', d.WD.min().item(), 'm (should be 0)')

# Check outflow ghost cells are NOT in node_BC
if hasattr(d, 'node_BC_outflow'):
    overlap = set(d.node_BC.tolist()) & set(d.node_BC_outflow.tolist())
    print(f'\nOutflow ghost cells in node_BC (should be 0): {len(overlap)}')
    print(f'node_BC_outflow: {d.node_BC_outflow.tolist()[:5]} ... ({len(d.node_BC_outflow)} cells)')